In [0]:
"""
01_pipeline_validation.py

SMIP V2 Pipeline Validation

Validates the Bronze, Silver and Gold layers after
pipeline execution.

Author:
Sumanth Vempalle

Version:
2.2.0
"""

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
)

CATALOG = "smip_v2"
SCHEMA = "manufacturing"

# ============================================================
# Helper Function
# ============================================================

def validate_table(table_name: str):

    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"

    print("=" * 80)
    print(f"Validating: {full_table}")

    df = spark.table(full_table)

    row_count = df.count()

    print(f"Rows : {row_count:,}")

    print("\nSchema")
    df.printSchema()

    print("\nSample Data")
    df.show(
        5,
        truncate=False,
    )

    print()

    return row_count


# ============================================================
# Bronze
# ============================================================

bronze_tables = [

    "manufacturing_events",

]

# ============================================================
# Silver
# ============================================================

silver_tables = [

    "parsed_events",

    "operation_events",

    "quality_events",

    "material_events",

    "packaging_events",

    "work_order_events",

    "execution_events",

    "serial_number_events",

]

# ============================================================
# Gold
# ============================================================

gold_tables = [

    "machine_kpis",

    "quality_kpis",

    "production_kpis",

    "material_kpis",

    "packaging_kpis",

    "factory_dashboard",

]

# ============================================================
# Validation
# ============================================================

print("\n")
print("=" * 80)
print("SMIP V2 PIPELINE VALIDATION")
print("=" * 80)

validation_results = []

for table in bronze_tables + silver_tables + gold_tables:

    rows = validate_table(table)

    validation_results.append(

        (

            table,

            rows,

        )

    )

print("=" * 80)
print("SUMMARY")
print("=" * 80)

summary = spark.createDataFrame(

    validation_results,

    [

        "table_name",

        "row_count",

    ],

)

summary.show(
    truncate=False,
)

print("=" * 80)
print("Validation completed.")
print(current_timestamp())
print("=" * 80)

# ============================================================
# Duplicate Event Validation
# ============================================================

print("\nChecking duplicate event_id values...")

duplicates = (

    spark.table(
        f"{CATALOG}.{SCHEMA}.parsed_events"
    )

    .groupBy(
        "event_id"
    )

    .agg(
        count("*").alias("records")
    )

    .filter(
        col("records") > 1
    )

)

duplicate_count = duplicates.count()

print(f"Duplicate Event IDs : {duplicate_count}")

if duplicate_count > 0:

    duplicates.show(
        truncate=False,
    )

# ============================================================
# Null Validation
# ============================================================

print("\nChecking NULL event_id values...")

null_events = (

    spark.table(
        f"{CATALOG}.{SCHEMA}.parsed_events"
    )

    .filter(
        col("event_id").isNull()
    )

)

print(

    f"NULL Event IDs : {null_events.count()}"

)

print("\nPipeline validation completed successfully.")